---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">GenAI & Advanced Nets: Consumo de LLMs via API</h1>

<h3 align="center">PhD. Julles Mitoura</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
  <img src="https://img.shields.io/badge/OpenAI-412991?style=for-the-badge&logo=openai&logoColor=white"/>
</div>

---

In [ ]:
# Obs: se você não estiver utilizando um ambiente virtual, instale as bibliotecas conforme se apresenta abaixo
#%pip install -q -r requirements.txt

# pip é o gerenciador de pacotes do Python. Pense nele como o instalador oficial de libs Python.
# no notebook, usar %pip ... é ideal porque instala no mesmo ambiente do kernel em uso.

# -q: quiet
# -r: requirement file, indica ao pip para instalar os pacotes listados no arquivo requirements.txt

---

<div align="center">

## <span style="color:#1E90FF;">LLMs via API: Visão Geral</span>

</div>

Nas aulas anteriores, construímos e treinamos modelos de linguagem do zero — primeiro com PyTorch, depois apenas com NumPy. Agora daremos um passo diferente: **consumir modelos já treinados e muito mais poderosos por meio de uma API**.

Grandes Modelos de Linguagem (LLMs) como o GPT-4o da OpenAI possuem bilhões de parâmetros e foram treinados em enormes volumes de dados. Treiná-los do zero exigiria infraestrutura computacional inacessível para a maioria dos projetos. A alternativa prática é consumi-los via API: enviamos uma requisição HTTP com o texto de entrada e recebemos a resposta gerada pelo modelo.

O endpoint central para geração de texto é o **Chat Completions**, que opera sobre uma lista de mensagens — permitindo conversas com múltiplos turnos, instruções de sistema e controle fino sobre o comportamento do modelo por meio de **parâmetros de geração**.

---

<div align="center">

## <span style="color:#1E90FF;">Setup: Configuração do Cliente</span>

</div>

In [ ]:
import os
import time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Cliente OpenAI configurado com sucesso.")

---

<div align="center">

## <span style="color:#1E90FF;">Estrutura das Mensagens</span>

</div>

O Chat Completions recebe uma lista de mensagens (`messages`). Cada mensagem possui dois campos obrigatórios:

| Campo | Tipo | Descrição |
|---|---|---|
| `role` | string | Quem está falando: `system`, `user` ou `assistant` |
| `content` | string | O texto da mensagem |

### Papéis (`role`)

- **`system`** — Define o comportamento geral do modelo: personalidade, restrições, formato esperado. É processado antes de tudo e orienta como o modelo responde às mensagens seguintes.
- **`user`** — Representa a entrada do usuário: perguntas, instruções, prompts.
- **`assistant`** — Representa respostas anteriores do modelo. Usado para construir conversas com múltiplos turnos (histórico).

```
[ system ]  →  define o contexto e as regras
[ user   ]  →  envia a pergunta ou tarefa
[ assistant ] →  resposta anterior (opcional, para histórico)
[ user   ]  →  próxima pergunta
    ...
```

---

<div align="center">

## <span style="color:#1E90FF;">Primeira Chamada</span>

</div>

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Você é um assistente educacional conciso e direto."},
        {"role": "user",   "content": "O que é um Large Language Model? Responda em 3 linhas."}
    ]
)

print(response.choices[0].message.content)

### Anatomia da resposta

O objeto `response` retornado pela API contém muito mais do que apenas o texto gerado:

In [ ]:
print("Modelo usado     :", response.model)
print("Motivo de parada :", response.choices[0].finish_reason)
print("Tokens de entrada:", response.usage.prompt_tokens)
print("Tokens de saída  :", response.usage.completion_tokens)
print("Tokens totais    :", response.usage.total_tokens)

> **`finish_reason`** indica por que o modelo parou de gerar texto:
> - `stop` — concluiu naturalmente
> - `length` — atingiu o limite de tokens (`max_tokens`)
> - `content_filter` — conteúdo bloqueado por política de uso

---

<div align="center">

## <span style="color:#1E90FF;">Parâmetros de Geração</span>

</div>

Os parâmetros de geração controlam **como** o modelo produz texto. Ajustá-los corretamente faz a diferença entre uma resposta genérica e uma resposta precisa para o contexto da aplicação.

---

### `model` — Escolha do Modelo

Define qual LLM processa a requisição. A escolha envolve trade-offs entre **capacidade**, **velocidade** e **custo**.

| Modelo | Uso recomendado |
|---|---|
| `gpt-4o-mini` | Tarefas rotineiras, alto volume, baixo custo |
| `gpt-4o` | Tarefas complexas, raciocínio, análise |
| `o3-mini` | Raciocínio matemático e lógico (chain-of-thought interno) |

> **Regra prática:** comece com `gpt-4o-mini` e escale para modelos mais capazes somente se a qualidade das respostas não for suficiente.

In [ ]:
prompt_modelo = "Explique o conceito de overfitting em uma frase."

for modelo in ["gpt-4o-mini", "gpt-4o"]:
    resp = client.chat.completions.create(
        model=modelo,
        messages=[{"role": "user", "content": prompt_modelo}]
    )
    print(f"[{modelo}]")
    print(resp.choices[0].message.content)
    print(f"Tokens: {resp.usage.total_tokens}")
    print()

---

### `temperature` — Criatividade vs. Determinismo

Controla a **aleatoriedade** na seleção de tokens. Tecnicamente, escala os logits antes da aplicação do softmax:

$$P(\text{token}_i) = \frac{e^{\,z_i / T}}{\sum_j e^{\,z_j / T}}$$

onde $z_i$ é o logit do token $i$ e $T$ é a temperatura.

| Valor | Efeito | Cenário de uso |
|---|---|---|
| `0.0` | Determinístico — sempre o token mais provável | Classificação, extração de dados, código |
| `0.2 – 0.5` | Muito focado, pouca variação | Resumos, respostas factuais |
| `0.7 – 1.0` | Equilibrado — padrão da maioria dos usos | Chatbots, explicações |
| `1.5 – 2.0` | Alta criatividade, saídas imprevisíveis | Brainstorming, escrita criativa |

> **Atenção:** `temperature` e `top_p` não devem ser alterados simultaneamente — ajuste apenas um deles por vez.

In [ ]:
prompt_temp = "Escreva um título criativo para um artigo sobre redes neurais."

for temp in [0.0, 0.7, 1.5]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_temp}],
        temperature=temp
    )
    print(f"[temperature={temp}] {resp.choices[0].message.content.strip()}")

**Experimento — repetindo com `temperature=0`:**

Com temperatura zero, múltiplas execuções do mesmo prompt devem retornar a mesma resposta (ou respostas muito próximas). Com temperatura alta, a variação é grande mesmo entre execuções idênticas.

In [ ]:
print("=== temperature=0.0 (3 execuções) ===")
for i in range(3):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_temp}],
        temperature=0.0
    )
    print(f"  [{i+1}] {resp.choices[0].message.content.strip()}")

print()
print("=== temperature=1.5 (3 execuções) ===")
for i in range(3):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_temp}],
        temperature=1.5
    )
    print(f"  [{i+1}] {resp.choices[0].message.content.strip()}")

---

### `max_tokens` — Limite de Tokens na Saída

Define o número máximo de tokens que o modelo pode gerar na resposta. Não afeta a **qualidade** — apenas trunca a saída quando o limite é atingido.

> **O que é um token?** Aproximadamente ¾ de uma palavra em inglês ou um pouco menos em português. "inteligência artificial" ≈ 4–5 tokens. Uma página de texto ≈ 500–700 tokens.

| Valor | Efeito |
|---|---|
| Muito baixo (ex.: `20`) | Resposta truncada, `finish_reason = "length"` |
| Adequado | Resposta completa, `finish_reason = "stop"` |
| Muito alto | Sem impacto se o modelo termina antes — mas aumenta o custo máximo possível |

> **Nota:** para modelos de raciocínio (`o3`, `o4-mini`), o parâmetro equivalente é `max_completion_tokens`, pois ele inclui os tokens do raciocínio interno.

In [ ]:
prompt_tokens = "Explique como funciona o mecanismo de atenção em transformers."

for limite in [20, 80, 300]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_tokens}],
        max_tokens=limite
    )
    texto = resp.choices[0].message.content.strip()
    motivo = resp.choices[0].finish_reason
    print(f"[max_tokens={limite} | finish_reason={motivo}]")
    print(texto)
    print()

---

### `top_p` — Nucleus Sampling

Define o tamanho do conjunto de tokens candidatos à seleção. O modelo considera apenas os tokens que juntos somam probabilidade acumulada de `top_p`.

| Valor | Efeito |
|---|---|
| `1.0` (padrão) | Considera todos os tokens do vocabulário |
| `0.9` | Considera apenas os tokens que juntos representam 90% da probabilidade |
| `0.1` | Considera apenas os tokens mais prováveis (resposta muito focada) |

A diferença prática para `temperature` é sutil, mas importante:
- `temperature` escala **todos os logits** antes do softmax — muda a forma da distribuição inteira.
- `top_p` **filtra** os tokens de menor probabilidade após o softmax — mantém a distribuição original, mas só amostra do núcleo.

> **Boa prática:** ajuste `temperature` **ou** `top_p`, nunca os dois ao mesmo tempo.

In [ ]:
prompt_topp = "Descreva a sensação de aprender algo novo."

for tp in [0.1, 0.5, 1.0]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_topp}],
        top_p=tp,
        temperature=1.0
    )
    print(f"[top_p={tp}]")
    print(resp.choices[0].message.content.strip())
    print()

---

### `frequency_penalty` e `presence_penalty` — Controle de Repetição

Ambos os parâmetros desincentivam o modelo a repetir tokens, mas de formas distintas. Matematicamente, eles ajustam os logits antes da amostragem:

$$z_i^{\prime} = z_i - \alpha \cdot c_i - \beta \cdot \mathbb{1}[c_i > 0]$$

onde $c_i$ é o número de vezes que o token $i$ já apareceu, $\alpha$ é o `frequency_penalty` e $\beta$ é o `presence_penalty`.

| Parâmetro | Intervalo | Efeito |
|---|---|---|
| `frequency_penalty` | `-2.0` a `2.0` | Penaliza proporcionalmente à **frequência** — quanto mais o token apareceu, mais penalizado |
| `presence_penalty` | `-2.0` a `2.0` | Penaliza pela **presença** — qualquer token que apareceu uma vez já recebe a penalidade máxima |

- **`frequency_penalty` alto:** evita que o modelo repita as mesmas palavras várias vezes no mesmo texto.
- **`presence_penalty` alto:** encoraja o modelo a mudar de assunto e introduzir novos temas.
- **Valores negativos:** efeito contrário — incentivam repetição (útil em raros casos de geração estruturada).

In [ ]:
prompt_penalty = "Fale sobre inteligência artificial."

configs = [
    {"label": "Sem penalidades (padrão)",          "frequency_penalty": 0.0, "presence_penalty": 0.0},
    {"label": "Alta frequency_penalty",             "frequency_penalty": 2.0, "presence_penalty": 0.0},
    {"label": "Alta presence_penalty",              "frequency_penalty": 0.0, "presence_penalty": 2.0},
]

for cfg in configs:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_penalty}],
        max_tokens=120,
        frequency_penalty=cfg["frequency_penalty"],
        presence_penalty=cfg["presence_penalty"]
    )
    print(f"[{cfg['label']}]")
    print(resp.choices[0].message.content.strip())
    print()

---

### `stop` — Sequências de Parada

Define até 4 strings que, quando geradas, fazem o modelo parar imediatamente. Útil para formatos estruturados onde se sabe exatamente onde a saída deve terminar.

A sequência de parada **não é incluída** na resposta.

In [ ]:
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Responda somente com a lista solicitada, sem introdução."},
        {"role": "user",   "content": "Liste 5 aplicações de machine learning, uma por linha, numeradas."}
    ],
    stop=["3."]  # para na 3ª linha — retorna apenas os 2 primeiros itens
)

print(resp.choices[0].message.content)
print(f"\nfinish_reason: {resp.choices[0].finish_reason}")

---

### `n` — Múltiplas Respostas

Gera `n` respostas independentes para o mesmo prompt. Cada resposta fica em `response.choices[i]`.

> **Custo:** o custo é proporcional a `n` — gerar 3 respostas consome aproximadamente 3x os tokens de saída. Use com moderação.

In [ ]:
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Sugira um nome para uma startup de IA educacional."}],
    n=3,
    temperature=1.2
)

for i, choice in enumerate(resp.choices):
    print(f"[Opção {i+1}] {choice.message.content.strip()}")

print(f"\nTotal de tokens consumidos: {resp.usage.total_tokens}")

---

### `seed` — Reprodutibilidade

Quando fornecido, o modelo tenta gerar a mesma resposta para o mesmo prompt com os mesmos parâmetros. Útil para **testes**, **debugging** e **experimentos comparativos**.

> **Importante:** a OpenAI não garante reprodutibilidade 100% — atualizações do modelo podem alterar as respostas mesmo com o mesmo `seed`. Verifique o campo `system_fingerprint` na resposta para identificar mudanças na infraestrutura.

In [ ]:
prompt_seed = "Gere um número aleatório entre 1 e 100 e justifique sua escolha."

print("=== Com seed=42 (3 execuções) ===")
for i in range(3):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_seed}],
        temperature=1.0,
        seed=42
    )
    print(f"  [{i+1}] {resp.choices[0].message.content.strip()[:80]}...")
    print(f"       system_fingerprint: {resp.system_fingerprint}")

print()
print("=== Sem seed (3 execuções) ===")
for i in range(3):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_seed}],
        temperature=1.0
    )
    print(f"  [{i+1}] {resp.choices[0].message.content.strip()[:80]}...")

---

### `stream` — Respostas em Tempo Real

Quando `stream=True`, os tokens são enviados progressivamente à medida que são gerados — em vez de aguardar a resposta completa. O cliente recebe um iterador de `chunks`, cada um contendo um fragmento do texto.

**Quando usar:**
- Interfaces conversacionais (chatbots, terminais)
- Respostas longas onde a latência percebida importa
- Monitoramento em tempo real da geração

**Quando não usar:**
- Pipelines de processamento em batch (não há ganho real)
- Quando você precisa do texto completo antes de processá-lo

In [ ]:
print("Gerando resposta em streaming:\n")

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Você é um professor paciente e detalhista."},
        {"role": "user",   "content": "Explique o que é backpropagation para um iniciante."}
    ],
    max_tokens=200,
    stream=True
)

resposta_completa = ""
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
        resposta_completa += delta

print(f"\n\nTotal de caracteres gerados: {len(resposta_completa)}")

---

<div align="center">

## <span style="color:#1E90FF;">Conversas com Múltiplos Turnos</span>

</div>

A API é **stateless** — ela não guarda histórico entre chamadas. Para simular uma conversa contínua, você deve enviar todo o histórico de mensagens em cada requisição.

In [ ]:
def chat(historico, mensagem_usuario, model="gpt-4o-mini", temperature=0.7):
    historico.append({"role": "user", "content": mensagem_usuario})

    resp = client.chat.completions.create(
        model=model,
        messages=historico,
        temperature=temperature
    )

    resposta = resp.choices[0].message.content
    historico.append({"role": "assistant", "content": resposta})

    return resposta, historico


historico = [
    {"role": "system", "content": "Você é um tutor de machine learning. Responda sempre de forma breve e com exemplos simples."}
]

perguntas = [
    "O que é uma rede neural?",
    "E o que são as camadas ocultas?",
    "Qual a diferença entre overfitting e underfitting?"
]

for pergunta in perguntas:
    print(f"[Usuário] {pergunta}")
    resposta, historico = chat(historico, pergunta)
    print(f"[Assistente] {resposta.strip()}")
    print(f"             (histórico: {len(historico)} mensagens)")
    print()

---

<div align="center">

## <span style="color:#1E90FF;">Boas Práticas</span>

</div>

### 1. System Prompt: invista na instrução de sistema

O `system` é o principal alavancador de qualidade. Um system prompt bem escrito reduz alucinações, enforça formato e elimina a necessidade de ajustes finos nos demais parâmetros.

**Elementos de um bom system prompt:**
- **Persona:** quem o modelo deve ser
- **Escopo:** o que pode e o que não pode responder
- **Formato:** como estruturar a resposta (markdown, JSON, listas...)
- **Tom:** formal, coloquial, técnico, educacional
- **Restrições:** idioma, tamanho, referências proibidas

In [ ]:
system_fraco = "Você é um assistente."

system_forte = """
Você é um assistente especializado em machine learning para estudantes de graduação.

Regras:
- Responda sempre em português brasileiro.
- Use linguagem acessível, sem jargões desnecessários.
- Quando relevante, inclua um exemplo prático curto.
- Limite suas respostas a no máximo 150 palavras.
- Se a pergunta não for sobre machine learning ou IA, diga educadamente que não pode ajudar com isso.
""".strip()

pergunta = "O que é regularização L2?"

for label, sysprompt in [("System fraco", system_fraco), ("System forte", system_forte)]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": sysprompt},
            {"role": "user",   "content": pergunta}
        ],
        temperature=0.5
    )
    print(f"[{label}]")
    print(resp.choices[0].message.content.strip())
    print()

### 2. Controle de Custos: monitore o uso de tokens

In [ ]:
# preços aproximados do gpt-4o-mini (verificar https://openai.com/pricing para valores atualizados)
PRECO_INPUT_POR_MILHAO  = 0.150  # USD
PRECO_OUTPUT_POR_MILHAO = 0.600  # USD

def estimar_custo(usage):
    custo_input  = (usage.prompt_tokens     / 1_000_000) * PRECO_INPUT_POR_MILHAO
    custo_output = (usage.completion_tokens / 1_000_000) * PRECO_OUTPUT_POR_MILHAO
    return custo_input + custo_output

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Você é um assistente conciso."},
        {"role": "user",   "content": "Quais são as principais diferenças entre CNN e RNN?"}
    ],
    max_tokens=200
)

uso = resp.usage
custo = estimar_custo(uso)

print(f"Tokens de entrada : {uso.prompt_tokens}")
print(f"Tokens de saída   : {uso.completion_tokens}")
print(f"Total de tokens   : {uso.total_tokens}")
print(f"Custo estimado    : USD ${custo:.6f}")
print()
print(resp.choices[0].message.content.strip())

### 3. Tratamento de Erros

A API pode retornar erros por diversas razões: limite de taxa (`RateLimitError`), chave inválida (`AuthenticationError`), conteúdo bloqueado (`BadRequestError`), indisponibilidade (`APIStatusError`). Trate-os explicitamente.

In [ ]:
from openai import RateLimitError, AuthenticationError, BadRequestError, APIStatusError

def chamar_api_com_retry(messages, model="gpt-4o-mini", max_tentativas=3, **kwargs):
    for tentativa in range(1, max_tentativas + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                **kwargs
            )
            return resp.choices[0].message.content

        except RateLimitError:
            espera = 2 ** tentativa
            print(f"  [Tentativa {tentativa}] Rate limit atingido. Aguardando {espera}s...")
            time.sleep(espera)

        except AuthenticationError:
            print("  Erro de autenticação: verifique a OPENAI_API_KEY no arquivo .env")
            return None

        except BadRequestError as e:
            print(f"  Requisição inválida: {e}")
            return None

        except APIStatusError as e:
            print(f"  Erro na API (status {e.status_code}): {e.message}")
            return None

    print("  Número máximo de tentativas atingido.")
    return None


resultado = chamar_api_com_retry(
    messages=[{"role": "user", "content": "Defina aprendizado por reforço em uma frase."}],
    temperature=0.3
)
print(resultado)

### 4. Saídas Estruturadas com JSON

Quando o modelo precisa retornar dados estruturados (para integração com outros sistemas), use `response_format` com `json_object` e instrua o modelo no system prompt.

In [ ]:
import json

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "Você extrai informações de textos e retorna sempre um JSON válido "
                "com os campos: conceito (string), area (string), dificuldade (string: basico|intermediario|avancado)."
            )
        },
        {
            "role": "user",
            "content": "Gradient descent é um algoritmo de otimização usado para minimizar funções de perda em modelos de machine learning."
        }
    ],
    response_format={"type": "json_object"},
    temperature=0.0
)

dados = json.loads(resp.choices[0].message.content)
print(json.dumps(dados, ensure_ascii=False, indent=2))

---

<div align="center">

## <span style="color:#1E90FF;">Resumo dos Parâmetros</span>

</div>

| Parâmetro | Padrão | Intervalo | Quando ajustar |
|---|---|---|---|
| `model` | — | modelos disponíveis | Sempre — escolha conforme custo e complexidade |
| `temperature` | `1.0` | `0.0` – `2.0` | Tarefas criativas (↑) ou determinísticas (↓) |
| `max_tokens` | sem limite | `1` – limite do modelo | Controlar custo máximo e evitar respostas longas |
| `top_p` | `1.0` | `0.0` – `1.0` | Alternativa ao `temperature` para focar nas palavras mais prováveis |
| `frequency_penalty` | `0.0` | `-2.0` – `2.0` | Evitar repetição de palavras no texto gerado |
| `presence_penalty` | `0.0` | `-2.0` – `2.0` | Encorajar diversidade de tópicos |
| `stop` | `null` | até 4 strings | Formatos estruturados com terminador conhecido |
| `n` | `1` | `≥ 1` | Gerar múltiplas alternativas para avaliação |
| `seed` | `null` | inteiro | Testes comparativos e reprodutibilidade |
| `stream` | `false` | `true/false` | Interfaces em tempo real |

---

### Guia rápido por tipo de aplicação

| Aplicação | Parâmetros recomendados |
|---|---|
| Chatbot educacional | `temperature=0.7`, system prompt detalhado |
| Extração de dados / classificação | `temperature=0.0`, `response_format=json_object` |
| Geração de código | `temperature=0.2`, `frequency_penalty=0.1` |
| Brainstorming / escrita criativa | `temperature=1.2–1.5`, `presence_penalty=0.5` |
| Resumo de documentos | `temperature=0.3`, `max_tokens` calibrado |
| Interface conversacional com streaming | `stream=True`, `max_tokens` com margem |
| Testes e experimentos A/B | `seed` fixo, `n≥2` |